# HW4: RAG Retrieval Evaluation

In this homework you'll wrap a retrieval pipeline and run it as a LangSmith experiment, pushing Recall@K and MRR metrics.

**What you'll learn:**
- How to wrap a retrieval function for LangSmith evaluation
- How to compute Recall@K and MRR as custom evaluators
- How to view and compare retrieval experiment results in LangSmith

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 1. Create the Retrieval Dataset

We'll create a small dataset of queries with known relevant documents. In a real scenario, this comes from the original course's retrieval solution.

In [ ]:
from langsmith import Client

client = Client()

# Sample retrieval evaluation dataset
retrieval_examples = [
    {
        "inputs": {"query": "How do I make chicken tikka masala?"},
        "outputs": {
            "expected_doc_ids": ["recipe_tikka_masala", "recipe_indian_curry_base", "recipe_tandoori_chicken"]
        },
    },
    {
        "inputs": {"query": "What are good vegan protein sources for cooking?"},
        "outputs": {
            "expected_doc_ids": ["guide_vegan_proteins", "recipe_tofu_stir_fry", "recipe_lentil_soup"]
        },
    },
    {
        "inputs": {"query": "Best way to bake sourdough bread"},
        "outputs": {
            "expected_doc_ids": ["recipe_sourdough", "guide_bread_baking", "guide_sourdough_starter"]
        },
    },
    {
        "inputs": {"query": "Quick weeknight pasta recipes"},
        "outputs": {
            "expected_doc_ids": ["recipe_aglio_olio", "recipe_cacio_pepe", "recipe_puttanesca"]
        },
    },
    {
        "inputs": {"query": "Gluten-free dessert ideas"},
        "outputs": {
            "expected_doc_ids": ["recipe_flourless_chocolate_cake", "recipe_macarons", "guide_gf_baking"]
        },
    },
]

dataset = client.create_dataset(
    dataset_name="recipe-retrieval-eval",
    description="Retrieval evaluation — queries with known relevant document IDs.",
)

for ex in retrieval_examples:
    client.create_example(
        dataset_id=dataset.id,
        inputs=ex["inputs"],
        outputs=ex["outputs"],
    )

print(f"Created dataset '{dataset.name}' with {len(retrieval_examples)} examples.")

## 2. Define a Mock Retrieval Pipeline

In a real setting you'd plug in your actual retriever (vector store, BM25, etc.).
Here we simulate one so the evaluation mechanics are clear.

> **Swap this out** with your actual retrieval function from the course.

In [ ]:
# Simulated document index — maps doc_id to content
DOC_INDEX = {
    "recipe_tikka_masala": "Chicken Tikka Masala: marinated chicken in creamy tomato sauce...",
    "recipe_indian_curry_base": "Indian curry base: onion, tomato, ginger, garlic, spices...",
    "recipe_tandoori_chicken": "Tandoori Chicken: yogurt-marinated chicken grilled at high heat...",
    "recipe_butter_chicken": "Butter Chicken: creamy tomato-based curry with tender chicken...",
    "guide_vegan_proteins": "Vegan protein sources: tofu, tempeh, seitan, lentils, chickpeas...",
    "recipe_tofu_stir_fry": "Tofu Stir-Fry: crispy tofu with vegetables in soy-ginger sauce...",
    "recipe_lentil_soup": "Red Lentil Soup: lentils simmered with cumin, turmeric, lemon...",
    "recipe_chickpea_curry": "Chickpea Curry: chickpeas in coconut-tomato sauce...",
    "recipe_sourdough": "Sourdough Bread: flour, water, salt, sourdough starter, long ferment...",
    "guide_bread_baking": "Bread Baking Guide: flour types, hydration, kneading, proofing...",
    "guide_sourdough_starter": "Sourdough Starter: feed flour and water daily for 7 days...",
    "recipe_focaccia": "Focaccia: olive oil-rich dough, dimpled, topped with herbs...",
    "recipe_aglio_olio": "Spaghetti Aglio e Olio: garlic, olive oil, chili flakes, parsley...",
    "recipe_cacio_pepe": "Cacio e Pepe: pecorino romano, black pepper, pasta water...",
    "recipe_puttanesca": "Puttanesca: tomatoes, olives, capers, anchovies, garlic...",
    "recipe_carbonara": "Carbonara: eggs, pecorino, guanciale, black pepper...",
    "recipe_flourless_chocolate_cake": "Flourless Chocolate Cake: chocolate, butter, eggs, sugar...",
    "recipe_macarons": "French Macarons: almond flour, egg whites, powdered sugar...",
    "guide_gf_baking": "Gluten-Free Baking Guide: almond flour, rice flour, xanthan gum...",
    "recipe_brownies": "Brownies: chocolate, butter, eggs, sugar, flour...",
}


def mock_retrieve(query: str, k: int = 5) -> list[str]:
    """Simulate retrieval — returns doc_ids ranked by 'relevance'.

    This is intentionally imperfect to make the evaluation interesting.
    """
    import random

    random.seed(hash(query) % 2**32)
    all_ids = list(DOC_INDEX.keys())
    random.shuffle(all_ids)

    # Simulate some signal: put 1-2 relevant docs near the top
    query_lower = query.lower()
    boosted = []
    rest = []
    for doc_id in all_ids:
        content = DOC_INDEX[doc_id].lower()
        # Simple keyword overlap
        if any(word in content for word in query_lower.split() if len(word) > 3):
            boosted.append(doc_id)
        else:
            rest.append(doc_id)

    ranked = boosted + rest
    return ranked[:k]

## 3. Define Retrieval Metrics

We'll compute **Recall@K** (fraction of relevant docs retrieved) and **MRR** (mean reciprocal rank of the first relevant doc).

In [ ]:
def recall_at_k(run, example) -> dict:
    """Recall@K: what fraction of expected docs were retrieved?"""
    run_outputs = run.outputs if hasattr(run, "outputs") else run.get("outputs", {}) or {}
    example_outputs = example.outputs if hasattr(example, "outputs") else example.get("outputs", {}) or {}

    retrieved = set(run_outputs.get("retrieved_doc_ids", []))
    expected = set(example_outputs.get("expected_doc_ids", []))

    if not expected:
        return {"score": 1.0, "comment": "No expected docs."}

    recall = len(retrieved & expected) / len(expected)
    return {
        "score": recall,
        "comment": f"Retrieved {len(retrieved & expected)}/{len(expected)} relevant docs.",
    }


def mrr(run, example) -> dict:
    """Mean Reciprocal Rank: 1/rank of the first relevant doc."""
    run_outputs = run.outputs if hasattr(run, "outputs") else run.get("outputs", {}) or {}
    example_outputs = example.outputs if hasattr(example, "outputs") else example.get("outputs", {}) or {}

    retrieved = run_outputs.get("retrieved_doc_ids", [])
    expected = set(example_outputs.get("expected_doc_ids", []))

    for i, doc_id in enumerate(retrieved):
        if doc_id in expected:
            return {"score": 1.0 / (i + 1), "comment": f"First relevant doc at rank {i + 1}."}

    return {"score": 0.0, "comment": "No relevant docs retrieved."}

## 4. Run the Retrieval Experiment

In [ ]:
from langsmith import evaluate


def run_retrieval(inputs: dict) -> dict:
    """Run the retrieval pipeline and return doc IDs."""
    query = inputs["query"]
    retrieved_ids = mock_retrieve(query, k=5)
    return {"retrieved_doc_ids": retrieved_ids}


results = evaluate(
    run_retrieval,
    data="recipe-retrieval-eval",
    evaluators=[recall_at_k, mrr],
    experiment_prefix="retrieval-baseline",
)

## 5. Run a Second Experiment (Larger K)

Let's see if retrieving more documents improves recall.

In [ ]:
def run_retrieval_k10(inputs: dict) -> dict:
    """Retrieve with k=10 instead of k=5."""
    query = inputs["query"]
    retrieved_ids = mock_retrieve(query, k=10)
    return {"retrieved_doc_ids": retrieved_ids}


results_k10 = evaluate(
    run_retrieval_k10,
    data="recipe-retrieval-eval",
    evaluators=[recall_at_k, mrr],
    experiment_prefix="retrieval-k10",
)

## 6. View and Compare Results (UI)

### Steps:
1. Go to your dataset **recipe-retrieval-eval** in LangSmith
2. Click the **Experiments** tab
3. You should see both experiments: `retrieval-baseline-*` and `retrieval-k10-*`
4. Select both and click **Compare** to see:
   - Recall@K scores side by side
   - MRR scores side by side
   - Which queries benefit most from a larger K

### What to look for:
- Does increasing K improve recall across the board?
- Does MRR stay the same? (It should — the first relevant doc's rank doesn't change)
- Which queries have the worst retrieval performance?